In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
train_check = pd.read_csv("../data/processed/train_clean.csv")
test_check = pd.read_csv("../data/processed/test_clean.csv")

print("Train shape:", train_check.shape)
print("Test shape:", test_check.shape)

print("Train missing:", train_check.isna().sum().sum())
print("Test missing:", test_check.isna().sum().sum())

Train shape: (635, 5752)
Test shape: (403, 5751)
Train missing: 597320
Test missing: 0


In [5]:
feature_cols = train_check.columns.drop(['id', 'target'])

In [6]:
print(train_check[feature_cols].dtypes.value_counts())

float64    5750
Name: count, dtype: int64


In [7]:
print("Minimum value:", train_check[feature_cols].min().min())
print("Maximum value:", train_check[feature_cols].max().max())

Minimum value: -962.4103917575974
Maximum value: 956.4318507137807


In [9]:
max_values = train_check[feature_cols].abs().max()

print(max_values.sort_values(ascending=False).head(10))

t178_arm_mag_z      962.410392
t179_arm_mag_z      943.930000
t180_arm_mag_z      930.107543
t177_arm_mag_z      925.936209
t143_arm_mag_z      907.087438
t181_arm_mag_z      898.869748
t144_arm_mag_z      893.772344
t142_arm_mag_z      886.075375
t176_arm_mag_z      881.081592
t202_ankle_mag_x    865.863626
dtype: float64


In [10]:
for sensor in ['acc', 'gyro', 'mag', 'ecg']:
    cols = [c for c in feature_cols if f'_{sensor}_' in c]

    print(
        sensor,
        "Min:", train_check[cols].min().min(),
        "Max:", train_check[cols].max().max()
    )

acc Min: -72.41244719306209 Max: 69.38230087903592
gyro Min: -6.1031945443068425 Max: 5.821070315704049
mag Min: -962.4103917575974 Max: 956.4318507137807
ecg Min: -10.56767014847952 Max: 10.575792191987867


In [11]:
missing_percent = (
    train_check[feature_cols].isna().mean() * 100
)

print(missing_percent.sort_values(ascending=False).head(10))

t157_arm_acc_y       17.165354
t137_arm_mag_z       17.165354
t113_ankle_mag_x     17.165354
t085_ankle_gyro_y    17.165354
t191_ankle_gyro_z    17.165354
t015_arm_acc_z       17.165354
t170_ankle_acc_y     17.165354
t193_arm_mag_x       17.165354
t114_arm_mag_z       17.165354
t199_arm_acc_z       17.165354
dtype: float64


In [12]:
print(missing_percent.sort_values().head(10))

t102_arm_gyro_x     15.118110
t047_chest_ecg_1    15.118110
t104_ankle_mag_z    15.118110
t176_chest_acc_z    15.118110
t044_ankle_mag_y    15.118110
t147_chest_acc_y    15.275591
t213_ankle_acc_y    15.275591
t015_arm_mag_x      15.275591
t140_ankle_acc_z    15.275591
t193_chest_ecg_2    15.275591
dtype: float64


In [14]:
X = train_check[feature_cols]
y = train_check['target']

In [15]:
print(y.value_counts())
print("\nPercentage:")
print((y.value_counts(normalize=True) * 100).round(2))

target
lying              60
cycling            59
jogging            59
walking            59
standing           58
sitting            58
running            58
climbing_stairs    54
arm_elevation      54
knees_bending      50
waist_bends        49
jump_front_back    17
Name: count, dtype: int64

Percentage:
target
lying              9.45
cycling            9.29
jogging            9.29
walking            9.29
standing           9.13
sitting            9.13
running            9.13
climbing_stairs    8.50
arm_elevation      8.50
knees_bending      7.87
waist_bends        7.72
jump_front_back    2.68
Name: proportion, dtype: float64


In [16]:
print("Infinite values:", np.isinf(X).sum().sum())

Infinite values: 0


In [17]:
print("Duplicate feature rows:", X.duplicated().sum())

Duplicate feature rows: 0


In [18]:
for value in [-9999, 9999, 99999]:
    print(value, (X == value).sum().sum())

-9999 0
9999 0
99999 0


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (508, 5750)
X_val: (127, 5750)
y_train: (508,)
y_val: (127,)


In [21]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

In [22]:
print("Training missing:", np.isnan(X_train_imputed).sum())
print("Validation missing:", np.isnan(X_val_imputed).sum())

print("Training shape:", X_train_imputed.shape)
print("Validation shape:", X_val_imputed.shape)

Training missing: 0
Validation missing: 0
Training shape: (508, 5750)
Validation shape: (127, 5750)


In [23]:
print("Total features:", len(feature_cols))

print("\nFirst 20 features:")
print(feature_cols[:20].tolist())

print("\nLast 20 features:")
print(feature_cols[-20:].tolist())

Total features: 5750

First 20 features:
['t000_chest_acc_x', 't000_chest_acc_y', 't000_chest_acc_z', 't000_chest_ecg_1', 't000_chest_ecg_2', 't000_ankle_acc_x', 't000_ankle_acc_y', 't000_ankle_acc_z', 't000_ankle_gyro_x', 't000_ankle_gyro_y', 't000_ankle_gyro_z', 't000_ankle_mag_x', 't000_ankle_mag_y', 't000_ankle_mag_z', 't000_arm_acc_x', 't000_arm_acc_y', 't000_arm_acc_z', 't000_arm_gyro_x', 't000_arm_gyro_y', 't000_arm_gyro_z']

Last 20 features:
['t249_chest_ecg_1', 't249_chest_ecg_2', 't249_ankle_acc_x', 't249_ankle_acc_y', 't249_ankle_acc_z', 't249_ankle_gyro_x', 't249_ankle_gyro_y', 't249_ankle_gyro_z', 't249_ankle_mag_x', 't249_ankle_mag_y', 't249_ankle_mag_z', 't249_arm_acc_x', 't249_arm_acc_y', 't249_arm_acc_z', 't249_arm_gyro_x', 't249_arm_gyro_y', 't249_arm_gyro_z', 't249_arm_mag_x', 't249_arm_mag_y', 't249_arm_mag_z']


In [24]:
parts = feature_cols.to_series().str.extract(r'^(t\d+)_(.+)$')

print("Unique time steps:", parts[0].nunique())
print("Unique sensor channels:", parts[1].nunique())

print("\nSensor channels:")
print(sorted(parts[1].dropna().unique()))

Unique time steps: 250
Unique sensor channels: 23

Sensor channels:
['ankle_acc_x', 'ankle_acc_y', 'ankle_acc_z', 'ankle_gyro_x', 'ankle_gyro_y', 'ankle_gyro_z', 'ankle_mag_x', 'ankle_mag_y', 'ankle_mag_z', 'arm_acc_x', 'arm_acc_y', 'arm_acc_z', 'arm_gyro_x', 'arm_gyro_y', 'arm_gyro_z', 'arm_mag_x', 'arm_mag_y', 'arm_mag_z', 'chest_acc_x', 'chest_acc_y', 'chest_acc_z', 'chest_ecg_1', 'chest_ecg_2']


In [26]:
X_test = test_check[feature_cols]

In [27]:
print("Test missing before:", X_test.isna().sum().sum())

Test missing before: 0


In [28]:
X_test_imputed = imputer.transform(X_test)

print(X_test_imputed.shape)
print(np.isnan(X_test_imputed).sum())

(403, 5750)
0


In [29]:
np.save("../data/processed/X_train_imputed.npy", X_train_imputed)
np.save("../data/processed/X_val_imputed.npy", X_val_imputed)
np.save("../data/processed/X_test_imputed.npy", X_test_imputed)

In [30]:
# Impute all 635 training samples using training medians
train_filled = pd.DataFrame(
    imputer.transform(train_check[feature_cols]),
    columns=feature_cols,
    index=train_check.index
)

# Impute test data using the same training medians
test_filled = pd.DataFrame(
    imputer.transform(test_check[feature_cols]),
    columns=feature_cols,
    index=test_check.index
)

In [31]:
train_final = pd.concat(
    [train_check[['id']], train_filled, train_check[['target']]],
    axis=1
)

test_final = pd.concat(
    [test_check[['id']], test_filled],
    axis=1
)

In [32]:
train_final.to_csv(
    "../data/processed/train_imputed.csv", index=False
)

test_final.to_csv(
    "../data/processed/test_imputed.csv", index=False
)

In [33]:
print("Train:", train_final.shape)
print("Test:", test_final.shape)

print("Train missing:", train_final.isna().sum().sum())
print("Test missing:", test_final.isna().sum().sum())

Train: (635, 5752)
Test: (403, 5751)
Train missing: 0
Test missing: 0
